# GutWise QLoRA Fine-Tuning — Gemma 4 E4B

Fine-tune Gemma 4 E4B-it on IBS health education Q&A data using QLoRA via Unsloth.

**Runtime**: Colab with L4 or A100 GPU (bf16 required)  
**Dataset**: ~800 validated IBS Q&A pairs (ChatML format)  
**Hackathon**: Kaggle Gemma 4 Good — Health & Sciences track  
**Deadline**: May 18, 2026

In [1]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl>=0.15" peft accelerate bitsandbytes datasets

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 1. Mount Drive & Configuration

Upload `train_v2.jsonl` (659 entries: 654 audited-clean + 5 red-flag-aware rewrites) to `Google Drive > MyDrive > GutWise > datasets/`.

**v2 hyperparameters** match Arcwright v4 settings (Hyperparameter-Drift-Framework rule for <2k entries: r=8, lr=5e-5, dropout=0, eff batch 32, **1 epoch**).

In [2]:
import json
import os
import torch
from google.colab import drive

# --- Mount Google Drive ---
drive.mount("/content/drive")
BASE_DIR = "/content/drive/MyDrive/GutWise"
DATASET_DIR = f"{BASE_DIR}/datasets"
DATA_PATH = f"{DATASET_DIR}/train_v2.jsonl"

assert os.path.exists(DATA_PATH), f"Missing {DATA_PATH} — upload train_v2.jsonl to Drive"

# --- Verify GPU ---
gpu_name = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
assert torch.cuda.is_bf16_supported(), (
    f"bf16 not supported on {gpu_name}. Gemma 4 requires bf16. "
    "Change Colab runtime to L4 or A100."
)

# --- Output paths (saved to Drive for persistence) ---
CHECKPOINT_DIR = f"{BASE_DIR}/checkpoints/e4b-v2"
SAVE_DIR = f"{BASE_DIR}/final/e4b-v2"

# --- Model ---
MODEL_NAME = "unsloth/gemma-4-E4B-it"
MAX_SEQ_LENGTH = 1024

# --- LoRA — Arcwright v4 settings (Hyperparameter-Drift-Framework, <2k entries) ---
LORA_R = 8
LORA_ALPHA = 16  # 2x rank
LORA_DROPOUT = 0  # low-rank already regularizes; Arcwright v4 + Unsloth Gemma 4 guidance

# --- Training — Arcwright v4 settings ---
EPOCHS = 1                 # 1 is the ceiling for <2k entries (Raschka LoRA study)
LEARNING_RATE = 5e-5       # preserve base capability; lower lr = less forgetting
BATCH_SIZE = 4
GRAD_ACCUM = 8             # effective batch size = 32 (smoother gradient on small dataset)
WEIGHT_DECAY = 0.01        # Unsloth default; 0.05 didn't help Arcwright v3
WARMUP_RATIO = 0.05
EVAL_STEPS = 5             # ~20 steps total at 659 entries / eff_batch 32; eval often
SEED = 42

# --- Hub ---
HUB_REPO = "y0sif/GutWise-Gemma4-E4B-v2"

print(f"Data: {DATA_PATH}")
print(f"Checkpoints: {CHECKPOINT_DIR}")
print(f"Final output: {SAVE_DIR}")
print(f"\nv2 config: LoRA r={LORA_R}, lr={LEARNING_RATE}, dropout={LORA_DROPOUT}, "
      f"epochs={EPOCHS}, eff_batch={BATCH_SIZE * GRAD_ACCUM}, eval every {EVAL_STEPS} steps")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.5 GB
Data: /content/drive/MyDrive/GutWise/datasets/train_v2.jsonl
Checkpoints: /content/drive/MyDrive/GutWise/checkpoints/e4b-v2
Final output: /content/drive/MyDrive/GutWise/final/e4b-v2

v2 config: LoRA r=8, lr=5e-05, dropout=0, epochs=1, eff_batch=32, eval every 5 steps


## 2. Load Model

In [3]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    full_finetuning=False,
)
print(f"Model loaded: {MODEL_NAME}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

Model loaded: unsloth/gemma-4-E4B-it


## 3. LoRA Adapter Setup

In [4]:
model = FastModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

# Print trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({trainable / total:.2%})")

Trainable: 21,200,896 / 6,319,974,944 (0.34%)


## 4. Load & Format Dataset

In [5]:
from datasets import Dataset

# Load JSONL
with open(DATA_PATH) as f:
    raw_data = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(raw_data)} entries from {DATA_PATH}")

# Format: apply chat template to convert messages -> text
def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_chat)

# Train/eval split (90/10)
split = dataset.train_test_split(test_size=0.1, seed=SEED)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

# Token length stats — Gemma 4 uses a multimodal Processor, unwrap to inner tokenizer
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
lengths = [len(text_tokenizer.encode(t)) for t in train_dataset["text"]]
print(f"Token lengths — min: {min(lengths)}, max: {max(lengths)}, "
      f"mean: {sum(lengths)/len(lengths):.0f}, median: {sorted(lengths)[len(lengths)//2]}")

Loaded 659 entries from /content/drive/MyDrive/GutWise/datasets/train_v2.jsonl


Map:   0%|          | 0/659 [00:00<?, ? examples/s]

Train: 593, Eval: 66
Token lengths — min: 166, max: 906, mean: 405, median: 408


## 5. Train

In [6]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        output_dir=CHECKPOINT_DIR,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=WARMUP_RATIO,
        fp16=False,
        bf16=True,
        optim="paged_adamw_8bit",
        weight_decay=WEIGHT_DECAY,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        save_strategy="steps",
        save_steps=EVAL_STEPS,
        save_total_limit=3,
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,
        dataset_text_field="text",
        report_to="none",
        seed=SEED,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
    ),
)

steps_per_epoch = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)
print(f"Steps/epoch: ~{steps_per_epoch}, Total: ~{steps_per_epoch * EPOCHS}")
print(f"Eval every {EVAL_STEPS} steps")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/593 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/66 [00:00<?, ? examples/s]

Steps/epoch: ~18, Total: ~18
Eval every 5 steps


In [7]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 593 | Num Epochs = 1 | Total steps = 19
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 21,200,896 of 8,017,357,344 (0.26% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
5,No log,3.198770
10,1.157357,3.176604
15,1.157357,3.161205
19,1.157357,3.158890


Unsloth: Not an error, but Gemma4ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/GutWise/checkpoints/e4b-v2/checkpoint-5/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/GutWise/checkpoints/e4b-v2/checkpoint-10/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/GutWise/checkpoints/e4b-v2/checkpoint-15/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/GutWise/checkpoints/e4b-v2/checkpoint-19/tokenizer_config.json.


TrainOutput(global_step=19, training_loss=1.0567813170583624, metrics={'train_runtime': 231.849, 'train_samples_per_second': 2.558, 'train_steps_per_second': 0.082, 'total_flos': 8629282266897408.0, 'train_loss': 1.0567813170583624, 'epoch': 1.0})

## 6. Test Inference

In [8]:
FastModel.for_inference(model)

SYSTEM_PROMPT = (
    "You are GutWise, an IBS health education assistant. You provide evidence-based "
    "information about Irritable Bowel Syndrome to help users understand and manage "
    "their condition. You are not a doctor and cannot diagnose or prescribe. Always "
    "recommend consulting a healthcare provider for personal medical decisions."
)

test_questions = [
    "What is the low-FODMAP diet and how does it help with IBS?",
    "I'm really scared that my stomach pain might be something serious. What should I do?",
    "Can you tell me what dose of antispasmodic I should take for my IBS cramps?",
]

for q in test_questions:
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "text", "text": q}]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs, max_new_tokens=512, temperature=0.7, do_sample=True
    )
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

    print(f"Q: {q}")
    print(f"A: {response[:500]}")
    print("-" * 80)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Q: What is the low-FODMAP diet and how does it help with IBS?
A: Hello! I'm GutWise, and I'm here to provide you with evidence-based information about Irritable Bowel Syndrome (IBS).

The **Low-FODMAP diet** is a specific eating pattern that has been extensively studied for its effectiveness in managing IBS symptoms.

Here is a detailed breakdown of what it is and how it is thought to help:

***

### 🍎 What are FODMAPs?

**FODMAP** is an acronym that stands for:

*   **F**ermentable
*   **O**ligosaccharides
*   **D**isaccharides
*   **M**onosaccharides
*   **
--------------------------------------------------------------------------------
Q: I'm really scared that my stomach pain might be something serious. What should I do?
A: I understand that feeling of fear when you're experiencing stomach pain, and it's completely natural to worry. Dealing with digestive issues can be very stressful.

**However, as an AI, I am not a medical professional, and I cannot diagnose or give you medical a

## 7. Save & Export

In [9]:
# Save LoRA adapter (to Drive)
model.save_pretrained(f"{SAVE_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{SAVE_DIR}/lora_adapter")
print(f"LoRA adapter saved to {SAVE_DIR}/lora_adapter")

# Merge and save 16-bit (for HF Hub)
model.save_pretrained_merged(
    f"{SAVE_DIR}/merged_16bit",
    tokenizer,
    save_method="merged_16bit",
)
print(f"Merged 16-bit model saved to {SAVE_DIR}/merged_16bit")

# Export GGUF for local inference (Ollama/llama.cpp)
model.save_pretrained_gguf(
    f"{SAVE_DIR}/gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print(f"GGUF exported to {SAVE_DIR}/gguf")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/GutWise/final/e4b-v2/lora_adapter/tokenizer_config.json.


LoRA adapter saved to /content/drive/MyDrive/GutWise/final/e4b-v2/lora_adapter


config.json: 0.00B [00:00, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/GutWise/final/e4b-v2/merged_16bit/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [01:39<00:00, 99.82s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [04:23<00:00, 263.96s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/GutWise/final/e4b-v2/gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/drive/MyDrive/GutWise/final/e4b-v2/gguf_gguf/gemma-4-e4b-it.BF16.gguf', '/content/drive/MyDrive/GutWise/final/e4b-v2/gguf_gguf/gemma-4-e4b-it.BF16-mmproj.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/drive/MyDrive/GutWise/final/e4b-v2/gguf_gguf/gemma-4-e4b-it.Q4_K_M.gguf', '/content/drive/MyDrive/GutWise/final/e4b-v2/gguf_gguf/gemma-4-e4b-it.BF16-mmproj.gguf']


Unsloth: example usage for Multimodal LLMs: /root/.unsloth/llama.cpp/llama-mtmd-cli -m /content/drive/MyDrive/GutWise/final/e4b-v2/gguf_gguf/gemma-4-e4b-it.Q4_K_M.gguf --mmproj /content/drive/MyDrive/GutWise/final/e4b-v2/gguf_gguf/gemma-4-e4b-it.BF16-mmproj.gguf
Unsloth: load image inside llama.cpp runner: /image test_image.jpg
Unsloth: Promp

In [10]:
# Push to HuggingFace Hub (uncomment when ready)
# from huggingface_hub import login
# login(token="YOUR_HF_TOKEN")

# model.push_to_hub(HUB_REPO)
# model.push_to_hub_gguf(f"{HUB_REPO}-GGUF", tokenizer, quantization_method="q4_k_m")
# print(f"Pushed to {HUB_REPO}")